# In tất cả cột của `GSE131907_sample_by_gene.h5`

Notebook nhỏ để kiểm tra cấu trúc file H5: liệt kê các key/group bên trong
và in ra toàn bộ tên cột của từng bảng. Hỗ trợ 2 trường hợp:

- File được ghi bằng `pandas.HDFStore`/`to_hdf` (giống các notebook khác trong
  thư mục này, ví dụ `geo_lung_download.ipynb`) -> đọc bằng `pandas`.
- File là HDF5 thô (không phải do pandas ghi) -> đọc bằng `h5py`, coi tên các
  dataset/attribute là "cột".

Tên file `sample_by_gene` gợi ý bảng có dạng hàng = mẫu (sample), cột = gene.

In [ ]:
import os
import glob

FILE_NAME = "GSE131907_sample_by_gene.h5"

# Sua tay o day neu ban da biet duong dan chinh xac, vi du:
# H5_PATH = "/kaggle/input/data_labeling/GSE131907_sample_by_gene.h5"
H5_PATH = None

SEARCH_DIRS = [
    "/kaggle/input",      # Kaggle: dataset duoc mount read-only o day
    "/kaggle/working",
    "/content",           # Google Colab
    ".",
]


def find_file(name, search_dirs):
    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        matches = glob.glob(os.path.join(d, "**", name), recursive=True)
        if matches:
            return matches[0]
    return None


if H5_PATH is None:
    H5_PATH = find_file(FILE_NAME, SEARCH_DIRS)

if H5_PATH is None or not os.path.exists(H5_PATH):
    raise FileNotFoundError(
        f"Khong tim thay '{FILE_NAME}'. Hay gan truc tiep duong dan vao bien H5_PATH o tren."
    )

print("Dung file:", H5_PATH)
print("Kich thuoc:", os.path.getsize(H5_PATH) / 1e6, "MB")

## Cách 1: đọc bằng pandas (`HDFStore`)

Nếu file được ghi bằng pandas, cách này sẽ liệt kê từng key và in cột của bảng đó.

In [ ]:
import pandas as pd

read_with_pandas_ok = False

try:
    with pd.HDFStore(H5_PATH, mode="r") as store:
        keys = store.keys()
        print(f"So key trong file: {len(keys)}")
        for key in keys:
            df = store.get(key)
            print(f"\n=== key: {key} ===")
            print("shape:", df.shape)
            cols = list(df.columns)
            print(f"So cot: {len(cols)}")
            for c in cols:
                print(" ", c)
    read_with_pandas_ok = len(keys) > 0
except Exception as e:
    print("Khong doc duoc bang pandas.HDFStore:", repr(e))
    read_with_pandas_ok = False

## Cách 2 (dự phòng): đọc bằng `h5py`

Chỉ chạy nếu cách 1 ở trên không đọc được (file không phải do pandas ghi ra).
In toàn bộ cây group/dataset và, với mỗi dataset dạng bảng có `dtype` compound,
in luôn tên các trường (đó chính là các "cột").

In [ ]:
if not read_with_pandas_ok:
    import h5py

    def print_tree(name, obj):
        indent = "  " * name.count("/")
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}[dataset] {name}  shape={obj.shape}  dtype={obj.dtype}")
            # Neu dataset la kieu compound (co nhieu truong), in ten tung truong
            if obj.dtype.names:
                print(f"{indent}  cot: {list(obj.dtype.names)}")
        else:
            print(f"{indent}[group]   {name}")

    with h5py.File(H5_PATH, "r") as f:
        print("Cac key o cap cao nhat:", list(f.keys()))
        f.visititems(print_tree)

        # Mot so dinh dang (vd anndata/loom) luu ten cot/gene trong 1 dataset rieng
        for candidate in ["var/_index", "var/gene_name", "col_attrs/gene_names",
                          "columns", "gene_names"]:
            if candidate in f:
                vals = f[candidate][()]
                print(f"\nTim thay danh sach cot tai '{candidate}', so luong: {len(vals)}")
                for v in vals:
                    print(" ", v.decode() if isinstance(v, bytes) else v)

## Cấu trúc thực tế của file này

Chạy cách 2 ở trên cho kết quả: file có 4 dataset ở cấp cao nhất, không phải
định dạng AnnData/loom mà là 4 mảng "song song" theo hàng (mỗi mẫu 1 hàng):

- `X` — ma trận biểu hiện, shape `(208506, 29634)` = 208506 mẫu x 29634 gene
- `genes` — tên 29634 gene, đây chính là **tên cột** của `X`
- `samples` — id của 208506 mẫu, tương ứng **từng hàng** của `X`
- `stages` — nhãn giai đoạn của từng mẫu (song song với `samples`)

Cell dưới đọc trực tiếp theo cấu trúc này và in 5 dòng đầu (không tải hết
208506 hàng vào RAM — dùng slicing của `h5py` để chỉ đọc 5 hàng cần thiết).

In [ ]:
import h5py
import pandas as pd

N_HEAD = 5

with h5py.File(H5_PATH, "r") as f:
    n_samples, n_genes = f["X"].shape
    genes = f["genes"][()]                 # doc het ten gene (29634 phan tu, nhe)
    samples_head = f["samples"][:N_HEAD]   # chi doc 5 phan tu dau
    stages_head = f["stages"][:N_HEAD]
    X_head = f["X"][:N_HEAD, :]            # chi doc 5 dong dau, day du cot


def to_str_list(arr):
    return [v.decode() if isinstance(v, bytes) else v for v in arr]


genes = to_str_list(genes)
samples_head = to_str_list(samples_head)
stages_head = to_str_list(stages_head)

print(f"X shape: {n_samples} mau x {n_genes} gene")
print(f"Tong so cot (gene): {len(genes)}")
print("5 ten cot (gene) dau tien:", genes[:5])
print("5 sample dau tien:", samples_head)
print("5 stage dau tien:", stages_head)

df_head = pd.DataFrame(X_head, index=samples_head, columns=genes)
df_head.index.name = "sample"
print("\n5 dong dau (sample x gene):")
df_head

## Gộp từng tế bào thành từng mẫu (bệnh nhân/mô)

Nhìn kỹ `samples`, mỗi giá trị có dạng `<barcode 10x>_<ten_mau>`, ví dụ:

- `AAACCTGAGAAACCGC_LN_05` -> mẫu `LN_05`
- `AAACCTGAGAAACGCC_NS_13` -> mẫu `NS_13`
- `AAACCTGAGAAGGTGA_LUNG_N18` -> mẫu `LUNG_N18`

Tức là mỗi hàng của `X` hiện tại là **1 tế bào** (dữ liệu single-cell), 208506
hàng chỉ là 208506 tế bào chứ không phải 208506 bệnh nhân. Phần **đuôi sau
barcode** (barcode luôn là chuỗi `A/C/G/T`) mới là ID mẫu thật.

Cell dưới:
1. Tách đuôi đó ra làm `sample_id` cho toàn bộ 208506 tế bào (chỉ thao tác trên
   mảng `samples`/`stages`, không đụng tới `X` nên rất nhẹ).
2. Kiểm tra mỗi `sample_id` chỉ có đúng 1 giá trị `stage` (vì stage là thuộc
   tính của bệnh nhân/mẫu, không phải của từng tế bào).
3. Thống kê lại: số mẫu sau khi gộp, số tế bào/mẫu, phân bố stage theo **số
   mẫu** (khác với phân bố theo số tế bào ở trên).

In [ ]:
import re

with h5py.File(H5_PATH, "r") as f:
    all_cells = to_str_list(f["samples"][()])   # 208506 phan tu, chi la string nen nhe
    all_stages = to_str_list(f["stages"][()])

BARCODE_RE = re.compile(r"^[ACGT]+_(.+)$")


def extract_sample_id(cell_id):
    """'AAACCTGAGAAACCGC_LN_05' -> 'LN_05'. Neu khong khop pattern thi giu nguyen."""
    m = BARCODE_RE.match(cell_id)
    return m.group(1) if m else cell_id


meta = pd.DataFrame({
    "cell_id": all_cells,
    "sample_id": [extract_sample_id(c) for c in all_cells],
    "stage": all_stages,
})

# Stage la thuoc tinh cua mau/benh nhan -> moi sample_id phai chi co 1 stage duy nhat.
# Neu co mau nao lech thi phai kiem tra lai quy uoc dat ten truoc khi tin ket qua ben duoi.
stage_nunique = meta.groupby("sample_id")["stage"].nunique()
inconsistent = stage_nunique[stage_nunique > 1]
if len(inconsistent):
    print(f"CANH BAO: {len(inconsistent)} mau co nhieu stage khac nhau, can kiem tra lai:")
    print(inconsistent)
else:
    print("OK: moi sample_id chi ung voi dung 1 gia tri stage.")

cells_per_sample = meta.groupby("sample_id").size().sort_values(ascending=False)
sample_stage = meta.groupby("sample_id")["stage"].first()

print(f"\nSo te bao truoc khi gop : {len(meta)}")
print(f"So mau sau khi gop      : {meta['sample_id'].nunique()}")

print("\nSo te bao / mau (top 10 nhieu nhat):")
print(cells_per_sample.head(10))
print("\nThong ke so te bao / mau:")
print(cells_per_sample.describe())

print("\nPhan bo stage THEO SO TE BAO (nhu truoc, mot te bao tinh 1 don vi):")
print(meta["stage"].value_counts())

print("\nPhan bo stage THEO SO MAU (moi mau chi tinh 1 lan, khong le thuoc so te bao):")
print(sample_stage.value_counts())

## (Tuỳ chọn) Cộng dồn biểu hiện gene theo mẫu — pseudobulk

Muốn có ma trận "mẫu x gene" thật sự (mỗi hàng là trung bình biểu hiện của tất
cả tế bào thuộc mẫu đó) thì phải đọc qua toàn bộ `X`. Vì `X` quá lớn để load
1 lần (~24.7 GB ở float32), cell dưới đọc theo từng khối `CHUNK` hàng bằng
`h5py`, cộng dồn tổng biểu hiện + số tế bào cho mỗi mẫu, rồi mới chia ra
trung bình — RAM cần dùng chỉ cỡ 1 khối (`CHUNK x 29634`), không phải cả ma
trận.

Việc này đọc hết ~24.7 GB từ đĩa nên khá tốn thời gian. Mặc định
`RUN_PSEUDOBULK = False`, đổi thành `True` khi thực sự cần chạy.

In [ ]:
import numpy as np

RUN_PSEUDOBULK = False   # doi thanh True de thuc su tinh pseudobulk (ton thoi gian)
CHUNK = 5000             # so hang doc moi lan tu X

if RUN_PSEUDOBULK:
    unique_samples = sample_stage.index.tolist()          # thu tu mau co dinh
    sample_to_idx = {s: i for i, s in enumerate(unique_samples)}
    n_unique = len(unique_samples)

    group_idx_all = meta["sample_id"].map(sample_to_idx).to_numpy()

    sums = np.zeros((n_unique, n_genes), dtype=np.float64)
    counts = np.zeros(n_unique, dtype=np.int64)

    with h5py.File(H5_PATH, "r") as f:
        X = f["X"]
        for start in range(0, n_samples, CHUNK):
            end = min(start + CHUNK, n_samples)
            chunk = X[start:end, :]                # doc 1 khoi (CHUNK x n_genes)
            g_idx = group_idx_all[start:end]
            for gi in np.unique(g_idx):
                mask = g_idx == gi
                sums[gi] += chunk[mask].sum(axis=0)
                counts[gi] += mask.sum()
            print(f"  da xu ly {end}/{n_samples} te bao", end="\r")

    means = sums / counts[:, None]

    pseudobulk = pd.DataFrame(means, index=unique_samples, columns=genes)
    pseudobulk.insert(0, "stage", sample_stage.loc[pseudobulk.index].values)
    pseudobulk.insert(1, "n_cells", counts.astype(int))

    print(f"\nMa tran pseudobulk (mau x gene): {pseudobulk.shape}")
    display(pseudobulk.head())
else:
    print("RUN_PSEUDOBULK = False -> bo qua buoc cong don bieu hien gene theo mau.")
    print("Doi thanh True o tren neu can ma tran pseudobulk day du.")

## Xuất ra H5 theo đúng định dạng đầu vào của `fair-comparison/*`

Các notebook trong [fair-comparison/](fair-comparison/) (`bmo.ipynb`, `bpso.ipynb`,
`gwopso.ipynb`, `sga.ipynb`, ...) đều nạp dữ liệu theo cùng 1 quy ước
(`_load_geo_direct` trong `bmo.ipynb`):

```python
expr  = pd.read_hdf(h5_path, key=f"{GSE_ID}/expr")     # gene/probe x sample
pheno = pd.read_hdf(h5_path, key=f"{GSE_ID}/pheno")    # sample x metadata
y = pheno.loc[expr.columns, "stage_encoded"]
X = expr.T   # -> sample x gene
```

Tức là file H5 phải là `pandas.HDFStore` với đúng 2 key `<GSE_ID>/expr` và
`<GSE_ID>/pheno`, trong đó `pheno` bắt buộc có cột `stage_encoded` (số hoá theo
đúng bảng `STAGE_TO_CODE = {"Normal": 0, "I": 1, "II": 2, "III": 3, "IV": 4}`
dùng chung toàn dự án, xem `geo_lung_download.ipynb`).

`stages` gốc trong file này là chuỗi kiểu AJCC có hậu tố phân nhóm nhỏ
(`IA3`, `IA2`, `IA`, `IV`, ...) chứ chưa phải I/II/III/IV thuần, nên cell dưới
**chuẩn hoá lại bằng đúng logic `normalize_stage` của `geo_lung_download.ipynb`**
trước khi số hoá, để nhất quán với các bộ GEO/TCGA khác trong dự án.

In [ ]:
assert "pseudobulk" in globals(), (
    "Chua co bien 'pseudobulk'. Hay dat RUN_PSEUDOBULK = True o cell tren va chay lai truoc."
)

# ----- 1. Chuan hoa stage (copy logic normalize_stage tu geo_lung_download.ipynb) -----
ARABIC_TO_ROMAN = {"1": "I", "2": "II", "3": "III", "4": "IV"}
MISSING = {"", "na", "n/a", "nan", "none", "null", "unknown",
           "not available", "not reported", "---", "-", "."}


def normalize_stage(x):
    """'IA3' -> 'I', 'IV' -> 'IV', 'Normal' -> 'Normal'. Tra ve None neu khong doc duoc."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s in MISSING:
        return None
    if "normal" in s:
        return "Normal"
    s = re.sub(r"^.*stage\s*[:=]?\s*", "", s).strip()
    if s in MISSING:
        return None
    if re.match(r"0(?![0-9])", s):
        return "0"
    m = re.match(r"(iv|iii|ii|i)(?![ivx])", s)
    if m:
        return m.group(1).upper()
    m = re.match(r"([1-4])(?![0-9])", s)
    if m:
        return ARABIC_TO_ROMAN[m.group(1)]
    return None


STAGE_TO_CODE = {"Normal": 0, "I": 1, "II": 2, "III": 3, "IV": 4}

pseudobulk = pseudobulk.copy()
pseudobulk["stage_norm"] = pseudobulk["stage"].map(normalize_stage)

unmapped = pseudobulk[pseudobulk["stage_norm"].isna()]
if len(unmapped):
    print(f"CANH BAO: {len(unmapped)} mau khong chuan hoa duoc stage, se bi LOAI khoi file xuat ra:")
    print(unmapped[["stage", "n_cells"]])
else:
    print("OK: tat ca mau deu chuan hoa duoc stage.")

pseudobulk = pseudobulk[pseudobulk["stage_norm"].notna()].copy()
pseudobulk["stage_encoded"] = pseudobulk["stage_norm"].map(STAGE_TO_CODE).astype(int)

print("\nPhan bo stage sau chuan hoa (theo so mau):")
print(pseudobulk["stage_norm"].value_counts())

# ----- 2. Tach expr (gene x sample) / pheno (sample x metadata) dung schema fair-comparison -----
GSE_ID_OUT = "GSE131907"

# /kaggle/input chi doc -> khong the ghi file vao cung thu muc voi H5_PATH neu
# no nam trong /kaggle/input. Uu tien /kaggle/working (ghi duoc), fallback ve
# thu muc chua H5_PATH (neu ghi duoc) roi moi toi thu muc hien tai.
if os.path.isdir("/kaggle/working"):
    OUT_DIR = "/kaggle/working"
elif not H5_PATH.startswith("/kaggle/input"):
    OUT_DIR = os.path.dirname(H5_PATH) or "."
else:
    OUT_DIR = "."
OUT_H5_PATH = os.path.join(OUT_DIR, "GSE131907_pseudobulk.h5")

expr_out = pseudobulk[genes].T          # gene x sample, dung nhu _load_geo_direct mong doi
expr_out.index.name = "gene"

pheno_out = pseudobulk[["stage", "stage_norm", "stage_encoded", "n_cells"]].copy()
pheno_out.index.name = "sample_id"

# mode="a": neu file da ton tai (vi du ban muon gop chung voi geo_expr.h5 khac)
# thi CHI ghi de dung 2 key nay, khong xoa cac key GSE khac da co san trong file.
with pd.HDFStore(OUT_H5_PATH, mode="a") as store:
    store.put(f"{GSE_ID_OUT}/expr", expr_out)
    store.put(f"{GSE_ID_OUT}/pheno", pheno_out)

print(f"\nDa ghi: {OUT_H5_PATH}")
print(f"  key '{GSE_ID_OUT}/expr' : {expr_out.shape} (gene x sample)")
print(f"  key '{GSE_ID_OUT}/pheno': {pheno_out.shape} (sample x metadata)")
print(
    "\nDe dung trong cac notebook fair-comparison (bmo.ipynb, bpso.ipynb, ...): "
    f"doi GSE_ID = \"{GSE_ID_OUT}\" va tro GEO_DIR/h5_path ve file nay "
    "(hoac gop 2 key tren vao geo_expr.h5 hien co)."
)